# Notebook 4 — Model: running the screening solver hands-on

This notebook builds a small domain, runs the shallow-water solver, and writes canonical outputs. It uses the same public objects the CLI drives — `StructuredGrid`, `ShallowWaterSolver`, `make_synthetic_tidal_boundary`, and the output writers. See `../architecture/ARCHITECTURE.md` for the full design.


## Learning objectives

- Build an Arakawa C-grid and understand its staggered layout.
- Force it with a synthetic tidal boundary and integrate forward.
- Compute power density, enforce the CFL condition, and write the canonical outputs.


## 4.1 Imports

Everything below degrades gracefully if the `model` package is not importable.


In [1]:
import numpy as np

try:
    from model.grid import StructuredGrid
    from model.solver import ShallowWaterSolver
    from model.forcing import make_synthetic_tidal_boundary
    from model.utils import speed, power_density, cfl_timestep
    HAVE_MODEL = True
except ImportError as exc:
    HAVE_MODEL = False
    print('model package not importable:', exc)
    print('set PYTHONPATH=src or pip install -e .')


## 4.2 Build a grid

`StructuredGrid` stores η at cell centres, u on x-faces, and v on y-faces. Depth is positive down. We override the metre-based coordinates with degree-based ones so GeoTIFF output is valid.


In [2]:
if HAVE_MODEL:
    grid = StructuredGrid.from_uniform(nx=40, ny=25, dx=2000.0, dy=2000.0)
    lon1 = np.linspace(120.0, 123.0, grid.nx)
    lat1 = np.linspace(10.0, 12.5, grid.ny)
    grid.lon, grid.lat = np.meshgrid(lon1, lat1)
    grid.h[:] = 50.0
    grid.h_u[:] = 50.0
    grid.h_v[:] = 50.0
    grid.open_boundary[:, 0] = True
    grid.open_boundary[:, -1] = True
    print('nx, ny        :', grid.nx, grid.ny)
    print('dx, dy [m]    :', grid.dx, grid.dy)
    print('wet cells     :', int(grid.mask.sum()))
    print('open boundary :', int(grid.open_boundary.sum()), 'cells')


nx, ny        : 40 25
dx, dy [m]    : 2000.0 2000.0
wet cells     : 1000
open boundary : 50 cells


![Arakawa C-grid: η at cell centres, u on x-faces, v on y-faces.](images/fig_cgrid.png)


## 4.3 CFL condition

The explicit scheme is stable only if the time step respects the CFL limit. `cfl_timestep` computes it from the grid spacing and max depth; the config applies a safety factor.


In [3]:
if HAVE_MODEL:
    dt = cfl_timestep(grid.dx, grid.dy, grid.h_max, safety=0.5)
    print(f'CFL timestep ~ {dt:.1f} s  (h_max = {grid.h_max:.0f} m)')


CFL timestep ~ 31.9 s  (h_max = 50 m)


## 4.4 Force and integrate

`make_synthetic_tidal_boundary` builds an M2+S2 boundary; the solver prescribes η on the open-boundary cells and advances with the forward-backward scheme. We sample mean power and max speed inside a callback. This is a **short demo**, not a resource estimate — production runs last 15 days.


In [4]:
if HAVE_MODEL:
    bnd = make_synthetic_tidal_boundary(
        int(grid.open_boundary.sum()), amplitude=0.8, constituents=['M2', 'S2'])
    solver = ShallowWaterSolver(grid, cd=0.0025)
    solver.set_open_boundary_eta(bnd)

    state = {
        'power_sum': np.zeros(grid.shape),
        'speed_max': np.zeros(grid.shape),
        'n': 0,
    }

    def cb(s, step):
        if step % 10 == 0:
            state['power_sum'] += s.compute_power_density()
            np.maximum(state['speed_max'], speed(s.u, s.v),
                       out=state['speed_max'])
            state['n'] += 1
        return None

    solver.run(dt=30.0, duration=6 * 3600.0, callback=cb,
               progress_interval=1e9)
    power_mean = state['power_sum'] / max(state['n'], 1)
    print('samples     :', state['n'])
    print(f'max speed   : {state["speed_max"].max():.2f} m/s')
    print(f'max power   : {power_mean.max():.1f} W/m^2 (cell max)')


samples     : 72
max speed   : 0.90 m/s
max power   : 87.7 W/m^2 (cell max)


## 4.5 Write canonical outputs

The writers in `model.output` produce the same Cloud-Optimised GeoTIFFs and GeoJSON the real pipeline publishes. We write to `output/workshop-demo/` (gitignored).


In [5]:
from pathlib import Path
from model.output import (
    write_mean_power_geotiff,
    write_raster_geotiff,
    write_hotspots_geojson,
)

if HAVE_MODEL:
    root = next((c for c in (Path('.'), Path('../..')) if (c / 'src').is_dir()),
                Path('.'))
    out = root / 'output' / 'workshop-demo'
    out.mkdir(parents=True, exist_ok=True)

    power_layer = np.where(grid.mask, power_mean, np.nan)
    speed_layer = np.where(grid.mask, state['speed_max'], np.nan)
    write_mean_power_geotiff(grid, power_layer,
                             str(out / 'tidal_power_density.tif'))
    write_raster_geotiff(grid, speed_layer,
                         str(out / 'max_current_speed.tif'),
                         'max depth-averaged current speed (m/s)')
    write_hotspots_geojson(grid, power_mean, threshold=1.0,
                           path=str(out / 'hotspots.geojson'))
    print('wrote outputs to', out)


wrote outputs to ../../output/workshop-demo


In [6]:
if HAVE_MODEL:
    import json
    for f in sorted(out.iterdir()):
        print(f.name, f.stat().st_size, 'bytes')
    fc = json.loads((out / 'hotspots.geojson').read_text())
    print('hotspot features:', len(fc['features']))


hotspots.geojson 268311 bytes
max_current_speed.tif 3899 bytes
tidal_power_density.tif 3923 bytes
hotspot features: 950


## 4.6 Tests

The same physics is validated by the test suite (mass conservation, M2-forced channel, standing-wave period):

```bash
python -m pytest src/model/tests/
```


## Next

The model works end to end. Move to [Notebook 5 — web](5.web.ipynb) to explore the Flask API and the map interface.

---

[Index](README.md) · [← 3.general-workflow.ipynb](3.general-workflow.ipynb) · [5.web.ipynb →](5.web.ipynb)
